<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/GGUF_NEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## gguf-builder

In [1]:
import os
import shutil
import glob
import subprocess
import torch
from huggingface_hub import hf_hub_download, snapshot_download
from safetensors.torch import load_file, save_file

print("Step 1: Cloning llama.cpp and installing conversion dependencies...")
if not os.path.exists("llama.cpp"):
    subprocess.run(["git", "clone", "https://github.com/ggml-org/llama.cpp.git"], check=True)

subprocess.run(["pip", "install", "-q", "numpy", "sentencepiece", "protobuf", "safetensors", "torch"], check=True)

print("Step 2: Downloading NVIDIA base model weights and your Topo-CBP lm_head.pt...")
base_dir = snapshot_download(repo_id="nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16")
head_file = hf_hub_download(repo_id="frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb", filename="lm_head.pt")

unified_dir = "./nemotron_unified_gguf"
if os.path.exists(unified_dir):
    shutil.rmtree(unified_dir)
shutil.copytree(base_dir, unified_dir)

print("Step 3: Sanitizing tokenizer_config.json...")
tcfg_path = os.path.join(unified_dir, "tokenizer_config.json")
if os.path.exists(tcfg_path):
    import json
    with open(tcfg_path, "r") as f:
        tcfg = json.load(f)
    for k in ["extra_special_tokens", "added_tokens_decoder"]:
        if k in tcfg and isinstance(tcfg[k], list):
            tcfg[k] = {}
    with open(tcfg_path, "w") as f:
        json.dump(tcfg, f, indent=2)

print("Step 4: Merging certified lm_head.pt into base model safetensors...")
head_state = torch.load(head_file, map_location="cpu")
if isinstance(head_state, dict) and "weight" in head_state:
    head_tensor = head_state["weight"]
elif isinstance(head_state, torch.Tensor):
    head_tensor = head_state
else:
    head_tensor = list(head_state.values())[0]

for sf in glob.glob(os.path.join(unified_dir, "*.safetensors")):
    tensors = load_file(sf)
    if "lm_head.weight" in tensors:
        tensors["lm_head.weight"] = head_tensor
        save_file(tensors, sf)
        print(f"Successfully replaced lm_head.weight in {sf}")
        break

print("Step 5: Converting unified directory to F16 GGUF...")
cmd = [
    "python3", "llama.cpp/convert_hf_to_gguf.py",
    unified_dir,
    "--outfile", "./nemotron-3.5-bolt-topo-governed-F16.gguf",
    "--outtype", "f16"
]

result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError(f"GGUF conversion failed with exit code {result.returncode}")

print("\nSUCCESS: F16 GGUF model ready at ./nemotron-3.5-bolt-topo-governed-F16.gguf")

Step 1: Cloning llama.cpp and installing conversion dependencies...
Step 2: Downloading NVIDIA base model weights and your Topo-CBP lm_head.pt...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

lm_head.pt: reconstructing file:   0%|          |  0.00B / 1.34GB            

lm_head.pt: downloading bytes:           |  0.00B            

Step 3: Sanitizing tokenizer_config.json...
Step 4: Merging certified lm_head.pt into base model safetensors...
Step 5: Converting unified directory to F16 GGUF...

SUCCESS: F16 GGUF model ready at ./nemotron-3.5-bolt-topo-governed-F16.gguf


In [2]:
import subprocess
import os

print("Building llama-quantize utility...")
subprocess.run(["cmake", "-S", "llama.cpp", "-B", "llama.cpp/build"], check=True)
subprocess.run(["cmake", "--build", "llama.cpp/build", "--config", "Release", "-j4"], check=True)

quant_bin = "llama.cpp/build/bin/llama-quantize" if os.path.exists("llama.cpp/build/bin/llama-quantize") else "llama.cpp/build/bin/Release/llama-quantize"

print("Quantizing F16 GGUF to Q4_K_M...")
subprocess.run([
    quant_bin,
    "./nemotron-3.5-bolt-topo-governed-F16.gguf",
    "./nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf",
    "Q4_K_M"
], check=True)

print("\nSUCCESS: Final quantized model ready at ./nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf")

Building llama-quantize utility...
Quantizing F16 GGUF to Q4_K_M...

SUCCESS: Final quantized model ready at ./nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf


In [10]:
%%shell
./llama.cpp/build/bin/llama-cli \
    -m ./nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf \
    -st \
    -p "Explain how topological gradient locking prevents catastrophic forgetting." \
    -n 512 \
    --temp 0.7



Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/ 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11913-7aca0d2ea
model      : ./nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf
ftype      : Q4_K - Medium
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> Explain how topological gradient locking prevents catastrophic forgetting.

[Start thinking]

Here's a plan to structure the explanation:
1.  **Define the core concepts:** Catastrophic forgetting, Topological Gradient Locking (

In [11]:
%%shell
python3 llama.cpp/gguf-py/gguf/scripts/gguf_dump.py ./nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf

INFO:gguf-dump:* Loading: ./nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf
* File is LITTLE endian, script is running on a LITTLE endian host.
* Dumping 62 key/value pair(s)
      1: UINT32     |        1 | GGUF.version = 3
      2: UINT64     |        1 | GGUF.tensor_count = 720
      3: UINT64     |        1 | GGUF.kv_count = 59
      4: STRING     |        1 | general.architecture = 'gemma4'
      5: STRING     |        1 | general.type = 'model'
      6: INT32      |        1 | general.sampling.top_k = 64
      7: FLOAT32    |        1 | general.sampling.top_p = 0.949999988079071
      8: FLOAT32    |        1 | general.sampling.temp = 1.0
      9: STRING     |        1 | general.name = 'Nemotron_Unified_Gguf'
     10: STRING     |        1 | general.size_label = '7.5B'
     11: STRING     |        1 | general.license = 'other'
     12: STRING     |        1 | general.license.name = 'openmdw-1.1'
     13: STRING     |        1 | general.license.link = 'https://openmdw.ai/license/1-1/'

## hf

In [12]:
from google.colab import userdata
from huggingface_hub import HfApi, login

# Retrieve your token from Colab Secrets and configure the repo target
HF_TOKEN = userdata.get('HF_TOKEN')
user_or_name = 'frankmorales2020'
repo_name = 'nemotron-3.5-bolt-e4b-topo-governed-fineweb-gguf'
repo_id = f"{user_or_name}/{repo_name}"

# Authenticate programmatically
login(token=HF_TOKEN)

api = HfApi()

# Ensure the repository exists (safe if it already exists)
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

print(f"Uploading quantized GGUF model to {repo_id}...")
api.upload_file(
    path_or_fileobj="./nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf",
    path_in_repo="nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf",
    repo_id=repo_id,
    repo_type="model",
)

print("Upload complete!")

Uploading quantized GGUF model to frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb-gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...topo-governed-Q4_K_M.gguf:   1%|          | 32.1MB / 5.34GB            

Validating                    :           |   0%            

Upload complete!


## inference-hf

In [2]:
!pip install -q llama-cpp-python

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

print("Downloading quantized GGUF model from Hugging Face...")
model_path = hf_hub_download(
    repo_id="frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb-gguf",
    filename="nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf"
)

print("Initializing local CPU runtime...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=4,
    verbose=False
)

print("\nGenerating response...")
response = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "Explain how topological gradient locking prevents catastrophic forgetting."}
    ],
    max_tokens=1024,
    temperature=0.7
)

print("\n" + response['choices'][0]['message']['content'])

Initializing local CPU runtime...


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024



Generating response...

Topological Gradient Locking (TGL) is a regularization technique designed to combat **catastrophic forgetting** in continual learning models. It achieves this by imposing structural constraints on the parameter updates during sequential task learning, ensuring that the knowledge acquired from previous tasks is preserved in a topologically stable manner.

Here is a detailed explanation of how TGL works and why it prevents catastrophic forgetting:

---

### 1. The Problem: Catastrophic Forgetting

In standard sequential learning, when a model ($\text{Model}_T$) is trained on Task $T$, its weights are updated to optimize for Task $T$'s objective function ($\mathcal{L}_T$). These updates often drastically alter the parameters that were critical for previously learned tasks ($\text{Task}_{1}, \text{Task}_{2}, \dots, \text{Task}_{T-1}$), leading to a severe performance drop on those older tasks. This forgetting is not due to insufficient data, but due to the *optimiz